In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import random
import math
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import mvit_v2_s, MViT_V2_S_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


c:\Users\Toranj\Desktop\AIM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
Device: cuda


In [ ]:


# meta data 
METADATA_CSV = Path(r"nexar_data\metadata.csv")

# extracted frames 
FRAME_DIR = Path(r"test\data\AIM_new_3lc\frames")

ID_COL = "video_id"
EVENT_COL = "time_of_event"

NUM_FRAMES = 16
IMAGE_SIZE = 224
BATCH_SIZE = 4
NUM_EPOCHS = 15
PATIENCE = 3
UNFREEZE_EPOCH = 5
SEED = 44

# Keep the original notebook's pretrained MViT-v2-S initialization.
PRETRAINED = True

CHECKPOINT_PATH = Path("nexar_data/best_mvit_collision_from_jpg_frames.pth")
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
def set_seed(seed=44):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
df = pd.read_csv(METADATA_CSV)
df["video_id"] = df[ID_COL].astype(str)

if "label" in df.columns:
    if df["label"].dtype == object:
        label_map = {
            "negative": 0, "no_collision": 0, "0": 0,
            "positive": 1, "collision": 1, "1": 1
        }
        mapped = df["label"].astype(str).str.strip().str.lower().map(label_map)
        
        if mapped.isna().any():
            numeric = pd.to_numeric(df["label"], errors="coerce")
            mapped = mapped.fillna(numeric)
        
        if mapped.isna().any():
            raise ValueError("Could not convert every label to 0/1.")
        df["label"] = mapped.astype(int)
    else:
        df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)
elif EVENT_COL in df.columns:
    df["label"] = pd.to_numeric(df[EVENT_COL], errors="coerce").notna().astype(int)




df = df.drop_duplicates("video_id").reset_index(drop=True)

IMAGE_EXTENSIONS = {".jpg"}

def list_frame_files(video_id):
    folder = FRAME_DIR / str(video_id)
    if not folder.is_dir():
        return []
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


df["frame_dir"] = df["video_id"].map(lambda vid: str(FRAME_DIR / vid))
df["frame_count"] = df["video_id"].map(lambda vid: len(list_frame_files(vid)))


before = len(df)
df = df[df["frame_count"] > 0].copy().reset_index(drop=True)


#### logggggs
print(f"Metadata rows: {before}")
print(f"Videos with extracted frames: {len(df)}")
print("\nClass distribution:")
print(df["label"].value_counts().sort_index())


df[["video_id", "label", "frame_count", "frame_dir"]].head()


Metadata rows: 1500
Videos with extracted frames: 1500

Class distribution:
label
0    750
1    750
Name: count, dtype: int64


,video_id,label,frame_count,frame_dir
0,video_000000,0,1200,test\data\AIM_new_3lc\frames\video_000000
1,video_000001,0,1204,test\data\AIM_new_3lc\frames\video_000001
2,video_000002,0,1204,test\data\AIM_new_3lc\frames\video_000002
3,video_000003,0,1147,test\data\AIM_new_3lc\frames\video_000003
4,video_000004,0,540,test\data\AIM_new_3lc\frames\video_000004


In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)

val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)


### logggggs
for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(part)} videos")
    print(part["label"].value_counts().sort_index().to_dict())


Train: 1050 videos
{0: 525, 1: 525}
Validation: 225 videos
{0: 112, 1: 113}
Test: 225 videos
{0: 113, 1: 112}


In [ ]:
class NexarDatasetAug(Dataset):
    def __init__(self, dataframe, train=False, use_flip=False):
        self.data = dataframe.reset_index(drop=True)
        self.train = train
        self.use_flip = use_flip
        self.mean = torch.tensor([0.45, 0.45, 0.45]).view(3, 1, 1, 1)
        self.std = torch.tensor([0.225, 0.225, 0.225]).view(3, 1, 1, 1)

        # stored frames / loading frames
        self.frame_paths = []
        for _, row in self.data.iterrows():
            folder = Path(row["frame_dir"])
            paths = sorted(
                p for p in folder.iterdir()
                if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
            )
            self.frame_paths.append(paths)

    def __len__(self):
        return len(self.data)

    def _augment(self, x):
        C, T, H, W = x.shape

        # horizantal flip is applied to the complete clip
        if self.use_flip and random.random() < 0.5:
            x = torch.flip(x, dims=[3])

        # random resized 
        if random.random() < 0.5:
            scale = random.uniform(0.85, 1.0)
            ratio = random.uniform(0.9, 1.1)
            area = scale * H * W

            w = min(W, int(round(math.sqrt(area * ratio))))
            h = min(H, int(round(math.sqrt(area / ratio))))

            top = random.randint(0, H - h)
            left = random.randint(0, W - w)

            x = x[:, :, top:top+h, left:left+w]
            x = F.interpolate(x.permute(1, 0, 2, 3), size=(H, W), mode="bilinear", align_corners=False).permute(1, 0, 2, 3)


        if random.random() < 0.5:
            x = x * random.uniform(0.85, 1.15)

        # contrast
        if random.random() < 0.5:
            contrast = random.uniform(0.85, 1.15)
            mean = x.mean()
            x = (x - mean) * contrast + mean

        # saturation
        if random.random() < 0.3:
            saturation = random.uniform(0.8, 1.2)
            gray = 0.299 * x[0:1] + 0.587 * x[1:2] + 0.114 * x[2:3]
            x = gray + saturation * (x - gray)

        return x.clamp(0.0, 1.0)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        paths = self.frame_paths[idx]

        # sample frame numbers from clips
        positions = np.linspace(0, len(paths) - 1, NUM_FRAMES).astype(int)

        frames = []
        for pos in positions:

            with Image.open(paths[pos]) as image:
                image = image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
                array = np.asarray(image).copy()

            frame = torch.from_numpy(array).permute(2, 0, 1).float() / 255.0
            frames.append(frame)


        # [T, C, H, W] -> [C, T, H, W]
        video = torch.stack(frames).permute(1, 0, 2, 3).contiguous()

        if self.train:
            video = self._augment(video)

        video = (video - self.mean) / self.std
        label = torch.tensor(int(row["label"]), dtype=torch.long)

        return video, label


In [ ]:
# maikng the inpuit  data 
train_dataset = NexarDatasetAug(train_df, train=True, use_flip=False)
val_dataset = NexarDatasetAug(val_df, train=False, use_flip=False)
test_dataset = NexarDatasetAug(test_df, train=False, use_flip=False)

# loading the data on gpu
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())


#### loggggssss
videos, labels = next(iter(train_loader))
print("Video batch:", videos.shape)
print("Labels:", labels)
print("Expected:", (BATCH_SIZE, 3, NUM_FRAMES, IMAGE_SIZE, IMAGE_SIZE))


Video batch: torch.Size([4, 3, 16, 224, 224])
Labels: tensor([0, 0, 0, 0])
Expected: (4, 3, 16, 224, 224)


In [ ]:
# temporal attention / pool / cnn 

class MultiScaleTemporalPool(nn.Module):

    def __init__(self, feature_dim=768, num_heads=8, num_layers=2, dropout=0.2, num_frames=16):
        super().__init__()

        # spatial feature projection
        # average pooling + max pooling
        self.spatial_projection = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # learning temporal 
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_frames, feature_dim))
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

        # temporal transform
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feature_dim,
            nhead=num_heads,
            dim_feedforward=feature_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.temporal_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.transformer_norm = nn.LayerNorm(feature_dim)


        # multy temporal cnn
        branch_dim = feature_dim // 3
        self.conv3 = nn.Sequential(
            nn.Conv1d(feature_dim, branch_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(branch_dim),
            nn.GELU()
        )

        self.conv5 = nn.Sequential(
            nn.Conv1d(feature_dim, branch_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(branch_dim),
            nn.GELU()
        )

        self.conv7 = nn.Sequential(
            nn.Conv1d(feature_dim, branch_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(branch_dim),
            nn.GELU()
        )

        self.multiscale_fusion = nn.Sequential(
            nn.Conv1d(branch_dim * 3, feature_dim, kernel_size=1),
            nn.BatchNorm1d(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # attention
        self.transformer_attention = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.multiscale_attention = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # fuse transform + cnn representations
        self.attention_fusion = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def attention_pool(self, x, attention):
        weights = torch.softmax(attention(x), dim=1)
        return torch.sum(x * weights, dim=1)

    def forward(self, patch_tokens):

        # Input: [B, T, H, W, C]
        B, T, H, W, C = patch_tokens.shape

        avg_features = patch_tokens.mean(dim=(2, 3))
        max_features = patch_tokens.amax(dim=(2, 3))

        x = torch.cat([avg_features, max_features], dim=-1)

        # [B, T, 2C] -> [B, T, C]
        x = self.spatial_projection(x)


        # transformer branch
        transformer_input = x + self.pos_embedding[:, :T, :]
        

        transformer_features = self.temporal_transformer(transformer_input)
        transformer_features = self.transformer_norm(transformer_features)
        transformer_pooled = self.attention_pool(transformer_features, self.transformer_attention)

        # cnn branch
        conv_input = x.transpose(1, 2)

        conv3 = self.conv3(conv_input)
        conv5 = self.conv5(conv_input)
        conv7 = self.conv7(conv_input)

        multiscale_features = torch.cat([conv3, conv5, conv7], dim=1)
        multiscale_features = self.multiscale_fusion(multiscale_features)
        multiscale_features = multiscale_features.transpose(1, 2)
        multiscale_pooled = self.attention_pool(multiscale_features, self.multiscale_attention)

    

        attention_features = self.attention_fusion(torch.cat([transformer_pooled,multiscale_pooled], dim=1))

        # temporla statistics
        temporal_mean = x.mean(dim=1)
        temporal_max = x.amax(dim=1)

        # three 768-dimensional vectors
        return attention_features, temporal_mean, temporal_max
        




    

class NexarCollisionModel(nn.Module):

    def __init__(self, num_classes=2, pretrained=True, dropout=0.4):
        super().__init__()

        weights = MViT_V2_S_Weights.DEFAULT if pretrained else None
        
        self.backbone = mvit_v2_s(weights=weights)

        in_features = self.backbone.head[1].in_features

        # removeing original classification head
        self.backbone.head = nn.Identity()

        # multi temporal pooling
        self.temporal_pool = MultiScaleTemporalPool(
            feature_dim=768,
            num_heads=8,
            num_layers=2,
            dropout=0.2,
            num_frames=16
        )

    
        fusion_dim = in_features * 4

        self.classifier = nn.Sequential(
            nn.LayerNorm(fusion_dim),

            nn.Linear(fusion_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        # Input: [B, C, T, H, W]

        x = self.backbone.conv_proj(x)
        x = x.flatten(2).transpose(1, 2)

        x = self.backbone.pos_encoding(x)

        thw = (
            self.backbone.pos_encoding.temporal_size,
            *self.backbone.pos_encoding.spatial_size
        )

        # MViT transformer blocks
        for block in self.backbone.blocks:
            x, thw = block(x, thw)

        x = self.backbone.norm(x)

        cls_token = x[:, 0]
        patch_tokens = x[:, 1:]

        B = x.shape[0]
        T, H, W = thw
        C = x.shape[-1]

        # [B, T*H*W, C] -> [B, T, H, W, C]
        patch_tokens = patch_tokens.reshape(B, T, H, W, C)

        
        attention_features, temporal_mean, temporal_max = self.temporal_pool(patch_tokens)

        # Feature fusion
        features = torch.cat(
            [
                cls_token,
                attention_features,
                temporal_mean,
                temporal_max
            ],
            dim=1
        )

        return self.classifier(features)



model = NexarCollisionModel(num_classes=2, pretrained=True).to(device)

print(model)

NexarCollisionModel(
  (backbone): MViT(
    (conv_proj): Conv3d(3, 96, kernel_size=(3, 7, 7), stride=(2, 4, 4), padding=(1, 3, 3))
    (pos_encoding): PositionalEncoding()
    (blocks): ModuleList(
      (0): MultiscaleBlock(
        (norm1): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
        (attn): MultiscaleAttention(
          (qkv): Linear(in_features=96, out_features=288, bias=True)
          (project): Sequential(
            (0): Linear(in_features=96, out_features=96, bias=True)
          )
          (pool_q): Pool(
            (pool): Conv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), groups=96, bias=False)
            (norm_act): Sequential(
              (0): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
            )
          )
          (pool_k): Pool(
            (pool): Conv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 8, 8), p

C:\Users\Toranj\AppData\Local\Temp\ipykernel_26388\353547497.py:32: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


In [ ]:
def set_backbone_trainable(model, trainable):
    for p in model.backbone.parameters():
        p.requires_grad = trainable

    for p in model.temporal_pool.parameters():
        p.requires_grad = True

    for p in model.classifier.parameters():
        p.requires_grad = True


def build_optimizer(model, lr_head=2e-5, lr_top=5e-6, lr_low=1e-6, weight_decay=0.01, top_n=4):
    blocks = list(model.backbone.blocks)
    
    head_params = list(model.temporal_pool.parameters()) + list(model.classifier.parameters())
    top_params = [p for block in blocks[-top_n:] for p in block.parameters()]

    used = {id(p) for p in head_params + top_params}
    low_params = [p for p in model.backbone.parameters() if id(p) not in used]

    return AdamW(
        [
            {"params": low_params, "lr": lr_low},
            {"params": top_params, "lr": lr_top},
            {"params": head_params, "lr": lr_head},
        ],
        weight_decay=weight_decay
    )


model = NexarCollisionModel(num_classes=2, pretrained=PRETRAINED).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
val_criterion = nn.CrossEntropyLoss()

optimizer = build_optimizer(model)

# learning rate scheduler
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1,total_iters=1)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, NUM_EPOCHS - 1), eta_min=1e-7)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[1])

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

###### loggggsss
print("Model, optimizer and loaders are ready.")


Model, optimizer and loaders are ready.


C:\Users\Toranj\AppData\Local\Temp\ipykernel_26388\353547497.py:32: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


In [ ]:
best_val_auc = 0.0
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_f1": [],
    "val_f1": [],
    "train_acc": [],
    "val_acc": [],
    "val_auc": [],
}

set_backbone_trainable(model, False)


ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.99))

for epoch in range(NUM_EPOCHS):

    if epoch == UNFREEZE_EPOCH:
        set_backbone_trainable(model, True)
        print("Backbone unfrozen")

    # train model 
    model.train()
    train_loss_sum = 0.0
    train_count = 0
    train_preds, train_targets = [], []

    for videos, labels in train_loader:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(videos)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ema.update_parameters(model)

        batch_size = labels.size(0)
        train_loss_sum += loss.item() * batch_size
        train_count += batch_size

        train_preds.extend(outputs.argmax(dim=1).detach().cpu().tolist())
        train_targets.extend(labels.detach().cpu().tolist())

    train_loss = train_loss_sum / train_count
    train_acc = accuracy_score(train_targets, train_preds)
    train_f1 = f1_score(train_targets, train_preds, average="macro", zero_division=0)


    # test result on validation set
    model.eval()
    val_loss_sum = 0.0
    val_count = 0
    val_preds, val_targets, val_probs = [], [], []

    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                outputs = model(videos)
                loss = val_criterion(outputs, labels)

            batch_size = labels.size(0)
            val_loss_sum += loss.item() * batch_size
            val_count += batch_size

            val_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            val_targets.extend(labels.cpu().tolist())
            val_probs.extend(torch.softmax(outputs.float(), dim=1)[:, 1].cpu().tolist())



    val_loss = val_loss_sum / val_count
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)
    val_precision = precision_score(val_targets, val_preds, average="macro", zero_division=0)
    val_recall = recall_score(val_targets, val_preds, average="macro", zero_division=0)
    val_auc = roc_auc_score(val_targets, val_probs)


    # looooogs
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)

    scheduler.step()

    #### more logggggs 
    print(
        f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Train Macro F1: {train_f1:.4f} | "
        f"Val Macro F1: {val_f1:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )

    print(
        f"    Val Macro Precision: {val_precision:.4f} | "
        f"Val Macro Recall: {val_recall:.4f} | "
        f"LR backbone: {optimizer.param_groups[0]['lr']:.2e} | "
        f"LR head: {optimizer.param_groups[2]['lr']:.2e}"
    )

    # saving best result 
    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_auc": best_val_auc,
                "val_loss": val_loss,
                "val_f1": val_f1,
                "class_names": ["no_collision", "collision"],
            },
            CHECKPOINT_PATH
        )
        print(f"    Saved best model: {CHECKPOINT_PATH}")

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break


best = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
print(f"\nBest validation AUC: {best['best_val_auc']:.4f} " f"(epoch {best['epoch']})")


Epoch [01/15] Train Loss: 0.6968 | Val Loss: 0.6826 | Train Acc: 0.4581 | Val Acc: 0.6222 | Train Macro F1: 0.4579 | Val Macro F1: 0.5817 | Val AUC: 0.7019
    Val Macro Precision: 0.7036 | Val Macro Recall: 0.6236 | LR backbone: 1.00e-06 | LR head: 2.00e-05
    Saved best model: nexar_data\best_mvit_collision_from_jpg_frames.pth
Epoch [02/15] Train Loss: 0.6797 | Val Loss: 0.6197 | Train Acc: 0.5790 | Val Acc: 0.6756 | Train Macro F1: 0.5784 | Val Macro F1: 0.6601 | Val AUC: 0.8245
    Val Macro Precision: 0.7168 | Val Macro Recall: 0.6765 | LR backbone: 9.89e-07 | LR head: 1.98e-05
    Saved best model: nexar_data\best_mvit_collision_from_jpg_frames.pth
Epoch [03/15] Train Loss: 0.6525 | Val Loss: 0.5655 | Train Acc: 0.6162 | Val Acc: 0.7022 | Train Macro F1: 0.6161 | Val Macro F1: 0.6915 | Val AUC: 0.8329
    Val Macro Precision: 0.7369 | Val Macro Recall: 0.7031 | LR backbone: 9.55e-07 | LR head: 1.90e-05
    Saved best model: nexar_data\best_mvit_collision_from_jpg_frames.pth
Epoc

In [ ]:
#### runnig model on test data set

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

test_loss_sum = 0.0
test_count = 0
test_preds, test_targets, test_probs = [], [], []

with torch.no_grad():
    for videos, labels in test_loader:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(videos)
            loss = val_criterion(outputs, labels)

        batch_size = labels.size(0)
        test_loss_sum += loss.item() * batch_size
        test_count += batch_size

        test_preds.extend(outputs.argmax(dim=1).cpu().tolist())
        test_targets.extend(labels.cpu().tolist())
        test_probs.extend(torch.softmax(outputs.float(), dim=1)[:, 1].cpu().tolist())


#### loggggs
test_loss = test_loss_sum / test_count
test_accuracy = accuracy_score(test_targets, test_preds)
test_precision = precision_score(test_targets, test_preds, average="macro", zero_division=0)
test_recall = recall_score(test_targets, test_preds, average="macro", zero_division=0)
test_f1 = f1_score(test_targets, test_preds, average="macro", zero_division=0)
test_auc = roc_auc_score(test_targets, test_probs)

print("\nFinal Test Results")
print(f"Loss:      {test_loss:.4f}")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"Macro F1:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(test_targets, test_preds))

print("\nClassification Report:")
print(classification_report(test_targets, test_preds, target_names=["No collision", "Collision"], digits=4, zero_division=0))



Final Test Results
Loss:      0.3028
Accuracy:  0.8889
Precision: 0.8903
Recall:    0.8890
Macro F1:  0.8888
ROC-AUC:   0.9439

Confusion Matrix:
[[ 97  16]
 [  9 103]]

Classification Report:
              precision    recall  f1-score   support

No collision     0.9151    0.8584    0.8858       113
   Collision     0.8655    0.9196    0.8918       112

    accuracy                         0.8889       225
   macro avg     0.8903    0.8890    0.8888       225
weighted avg     0.8904    0.8889    0.8888       225

